# Flickr30k Baseline Preflight

Phase này kiểm định độc lập output `F30K_Data`, resolve lại đường dẫn
ảnh trên Kaggle và khóa protocol trước khi chạy song song sáu baseline.

## Input bắt buộc — đúng 2 input

1. Output notebook `F30K_Data`.
2. Dataset `Flickr Image dataset` chứa `flickr30k_images/*.jpg`.

Không gắn Karpathy split, MS COCO, checkpoint Teacher/Student hoặc output KD
ở phase này.

## Output

- `F30K_BASE_READY.json`;
- `configs/F30K_BASELINE_PROTOCOL.json`;
- `parquet/f30k_trainval_resolved.parquet`;
- `parquet/f30k_test_resolved_sealed.parquet`;
- `references/f30k_validation_references.json`;
- manifest SHA-256 đầy đủ.

Runtime: CPU, Accelerator None, Internet Off; dự kiến 3–10 phút.


## Cell 1 — Data Contract và SHA-256


In [1]:
import csv
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
OUTPUT_ROOT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT", "/kaggle/working/F30K_Base"
))
CONFIG_DIR = OUTPUT_ROOT / "configs"
PARQUET_DIR = OUTPUT_ROOT / "parquet"
REFERENCE_DIR = OUTPUT_ROOT / "references"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in (
    OUTPUT_ROOT, CONFIG_DIR, PARQUET_DIR, REFERENCE_DIR, MANIFEST_DIR
):
    directory.mkdir(parents=True, exist_ok=True)

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

contract_matches = sorted(
    INPUT_ROOT.rglob("F30K_DATA_CONTRACT.json")
)
assert len(contract_matches) == 1, (
    "Cần đúng một output F30K_Data", contract_matches
)
CONTRACT_PATH = contract_matches[0]
CONTRACT_ROOT = CONTRACT_PATH.parent
CONTRACT = read_json(CONTRACT_PATH)

assert CONTRACT["status"] == "complete"
assert CONTRACT["dataset"] == "Flickr30k"
assert CONTRACT["schema_version"] == "f30k_data_contract_v1"
assert CONTRACT["test_set_status"] == "sealed"
assert CONTRACT["test_caption_exported"] is False

ARTIFACTS = {}
for name, metadata in CONTRACT["artifacts"].items():
    path = CONTRACT_ROOT / metadata["relative_path"]
    assert path.is_file(), (name, path)
    assert path.stat().st_size == int(metadata["size_bytes"]), name
    assert sha256(path) == metadata["sha256"], name
    ARTIFACTS[name] = path

required = {
    "train", "validation", "image_manifest", "test_sealed", "vocabulary"
}
assert set(ARTIFACTS) == required, set(ARTIFACTS)

print("=" * 72)
print("F30K BASE CELL 1 — UPSTREAM CONTRACT")
print("Status                    : PASSED")
print("Contract                  :", CONTRACT_PATH)
print("Artifacts                 :", len(ARTIFACTS), "/ 5 VERIFIED")
print("Annotation SHA-256         :", CONTRACT["annotation_sha256"])
print("Vocabulary SHA-256         :", CONTRACT["vocabulary_sha256"])
print("Test status               : SEALED / CAPTION NOT EXPORTED")


F30K BASE CELL 1 — UPSTREAM CONTRACT
Status                    : PASSED
Contract                  : /kaggle/input/notebooks/thienphan0712/f30k-data/F30K_Data/F30K_DATA_CONTRACT.json
Artifacts                 : 5 / 5 VERIFIED
Annotation SHA-256         : db779ee2c5df40c25ff0eebeca464f6124b620cc4a9398650aea71fd1b53d4af
Vocabulary SHA-256         : c94727d24d24130126ae7c88d9537714b88ff3dde12ac741f9cd4a0757287a83
Test status               : SEALED / CAPTION NOT EXPORTED


## Cell 2 — Split, token và Test boundary


In [2]:
train_table = pq.read_table(ARTIFACTS["train"])
validation_table = pq.read_table(ARTIFACTS["validation"])
image_manifest_table = pq.read_table(ARTIFACTS["image_manifest"])
test_table = pq.read_table(ARTIFACTS["test_sealed"])
vocabulary = read_json(ARTIFACTS["vocabulary"])

assert train_table.num_rows == 145_000
assert validation_table.num_rows == 5_070
assert image_manifest_table.num_rows == 30_014
assert test_table.num_rows == 1_000
assert len(vocabulary["tokens"]) == 7_456
assert vocabulary["special_token_ids"] == {
    "pad": 0, "bos": 1, "eos": 2, "unk": 3
}

train_ids = set(train_table["image_id"].to_pylist())
validation_ids = set(validation_table["image_id"].to_pylist())
test_ids = set(test_table["image_id"].to_pylist())
assert (len(train_ids), len(validation_ids), len(test_ids)) == (
    29_000, 1_014, 1_000
)
assert not (train_ids & validation_ids)
assert not (train_ids & test_ids)
assert not (validation_ids & test_ids)

def caption_counts(table):
    counts = {}
    for image_id in table["image_id"].to_pylist():
        counts[int(image_id)] = counts.get(int(image_id), 0) + 1
    return counts

assert set(caption_counts(train_table).values()) == {5}
assert set(caption_counts(validation_table).values()) == {5}

for table in (train_table, validation_table):
    for token_ids in table["token_ids"].to_pylist():
        assert 2 <= len(token_ids) <= 31
        assert token_ids[0] == 1 and token_ids[-1] == 2
        assert min(token_ids) >= 0
        assert max(token_ids) < len(vocabulary["tokens"])

assert "caption" not in test_table.column_names
assert all(value is False for value in test_table["caption_included"].to_pylist())
assert set(test_table["test_set_status"].to_pylist()) == {"sealed"}

print("=" * 72)
print("F30K BASE CELL 2 — SPLIT AND TOKEN AUDIT")
print("Status                    : PASSED")
print("Images Train/Val/Test     : 29000 / 1014 / 1000")
print("Captions Train/Val        : 145000 / 5070")
print("Caption per image         : 5 / 5")
print("Vocabulary                : 7456")
print("Decoder positions         : <= 31")
print("Split overlap             : 0 / 0 / 0")
print("Test status               : SEALED / NO CAPTION COLUMN")


F30K BASE CELL 2 — SPLIT AND TOKEN AUDIT
Status                    : PASSED
Images Train/Val/Test     : 29000 / 1014 / 1000
Captions Train/Val        : 145000 / 5070
Caption per image         : 5 / 5
Vocabulary                : 7456
Decoder positions         : <= 31
Split overlap             : 0 / 0 / 0
Test status               : SEALED / NO CAPTION COLUMN


## Cell 3 — Resolve đường dẫn ảnh trong phiên hiện tại


In [3]:
jpg_candidates = [
    path for path in INPUT_ROOT.rglob("*.jpg")
    if "flickr" in path.as_posix().lower()
]
by_parent = {}
for path in jpg_candidates:
    by_parent.setdefault(path.parent, []).append(path)

complete_roots = sorted(
    (
        (parent, paths)
        for parent, paths in by_parent.items()
        if len({path.name for path in paths}) >= 31_000
    ),
    key=lambda item: (-len({path.name for path in item[1]}), item[0].as_posix()),
)
assert complete_roots, {
    parent.as_posix(): len({path.name for path in paths})
    for parent, paths in by_parent.items()
}
SELECTED_IMAGE_ROOT, selected_paths = complete_roots[0]
image_by_name = {}
for path in sorted(selected_paths, key=lambda value: value.as_posix()):
    assert path.name not in image_by_name, path.name
    image_by_name[path.name] = path

resolved_schema = pa.schema([
    pa.field("image_id", pa.int64(), nullable=False),
    pa.field("experiment_split", pa.string(), nullable=False),
    pa.field("image_filename", pa.string(), nullable=False),
    pa.field("image_path", pa.string(), nullable=False),
])
sealed_schema = pa.schema([
    pa.field("image_id", pa.int64(), nullable=False),
    pa.field("experiment_split", pa.string(), nullable=False),
    pa.field("image_filename", pa.string(), nullable=False),
    pa.field("image_path", pa.string(), nullable=False),
    pa.field("caption_included", pa.bool_(), nullable=False),
    pa.field("test_set_status", pa.string(), nullable=False),
])

resolved_rows = []
for row in image_manifest_table.to_pylist():
    filename = Path(row["image_path"]).name
    assert filename in image_by_name, filename
    resolved_rows.append({
        "image_id": int(row["image_id"]),
        "experiment_split": str(row["experiment_split"]),
        "image_filename": filename,
        "image_path": image_by_name[filename].as_posix(),
    })

test_rows = []
for row in test_table.to_pylist():
    filename = Path(row["image_path"]).name
    assert filename in image_by_name, filename
    test_rows.append({
        "image_id": int(row["image_id"]),
        "experiment_split": "test",
        "image_filename": filename,
        "image_path": image_by_name[filename].as_posix(),
        "caption_included": False,
        "test_set_status": "sealed",
    })

RESOLVED_PATH = PARQUET_DIR / "f30k_trainval_resolved.parquet"
TEST_RESOLVED_PATH = PARQUET_DIR / "f30k_test_resolved_sealed.parquet"
pq.write_table(
    pa.Table.from_pylist(resolved_rows, schema=resolved_schema), RESOLVED_PATH
)
pq.write_table(
    pa.Table.from_pylist(test_rows, schema=sealed_schema), TEST_RESOLVED_PATH
)
assert pq.ParquetFile(RESOLVED_PATH).metadata.num_rows == 30_014
assert pq.ParquetFile(TEST_RESOLVED_PATH).metadata.num_rows == 1_000

print("=" * 72)
print("F30K BASE CELL 3 — IMAGE PATH RESOLUTION")
print("Status                    : PASSED")
print("Selected image root       :", SELECTED_IMAGE_ROOT)
print("Resolved Train/Val        :", len(resolved_rows))
print("Resolved Test             :", len(test_rows), "SEALED")


F30K BASE CELL 3 — IMAGE PATH RESOLUTION
Status                    : PASSED
Selected image root       : /kaggle/input/datasets/hsankesara/flickr-image-dataset/flickr30k_images/flickr30k_images
Resolved Train/Val        : 30014
Resolved Test             : 1000 SEALED


## Cell 4 — Validation references và baseline authorization


In [4]:
validation_references = {}
annotations = []
for row in validation_table.select(
    ["annotation_id", "image_id", "caption"]
).to_pylist():
    image_id = int(row["image_id"])
    validation_references.setdefault(image_id, []).append(str(row["caption"]))
    annotations.append({
        "id": int(row["annotation_id"]),
        "image_id": image_id,
        "caption": str(row["caption"]),
    })

assert len(validation_references) == 1_014
assert set(map(len, validation_references.values())) == {5}
VALIDATION_REFERENCE_PATH = (
    REFERENCE_DIR / "f30k_validation_references.json"
)
write_json(VALIDATION_REFERENCE_PATH, {
    "info": {
        "description": "Flickr30k Karpathy Validation references only",
        "test_set_status": "sealed",
    },
    "images": [{"id": image_id} for image_id in sorted(validation_references)],
    "annotations": sorted(annotations, key=lambda row: row["id"]),
    "type": "captions",
})

PROTOCOL_PATH = CONFIG_DIR / "F30K_BASELINE_PROTOCOL.json"
protocol = {
    "schema_version": "f30k_baseline_protocol_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "dataset": "Flickr30k",
    "dataset_contract_sha256": sha256(CONTRACT_PATH),
    "annotation_sha256": CONTRACT["annotation_sha256"],
    "vocabulary_sha256": CONTRACT["vocabulary_sha256"],
    "vocabulary_size": 7_456,
    "max_decoder_positions": 31,
    "development_seed": 42,
    "confirmation_seeds": [123, 2026],
    "maximum_epochs": 20,
    "selection_metric": "Validation CIDEr-D",
    "minimum_cider_improvement": 0.001,
    "early_stopping": {
        "enabled": True,
        "minimum_completed_epochs": 5,
        "patience": 4,
    },
    "generation": {"beam_size": 3, "length_penalty": 0.7},
    "baseline_models": {
        "T1": "ExpansionNet v2",
        "T2": "BLIP Base",
        "T3": "SigLIP ViT-B/16",
        "S1": "MobileNetV3-Large",
        "S2": "MobileViT-S",
        "S3": "TinyViT-5M",
    },
    "test_set_status": "sealed",
    "test_set_used": False,
    "test_caption_exported": False,
}
write_json(PROTOCOL_PATH, protocol)

READY_PATH = OUTPUT_ROOT / "F30K_BASE_READY.json"
ready = {
    "schema_version": "f30k_base_ready_v1",
    "created_at_utc": now_utc(),
    "status": "authorized",
    "authorized_baselines": ["T1", "T2", "T3", "S1", "S2", "S3"],
    "parallel_execution_authorized": True,
    "development_seed": 42,
    "validation_reference_count": len(annotations),
    "data_contract_sha256": sha256(CONTRACT_PATH),
    "protocol_sha256": sha256(PROTOCOL_PATH),
    "resolved_trainval_sha256": sha256(RESOLVED_PATH),
    "resolved_test_sha256": sha256(TEST_RESOLVED_PATH),
    "validation_references_sha256": sha256(VALIDATION_REFERENCE_PATH),
    "upstream_manifest_note": (
        "image_manifest verified from Data Contract artifact map; "
        "upstream summary manifest omitted that filename"
    ),
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(READY_PATH, ready)

artifact_paths = sorted(
    path for path in OUTPUT_ROOT.rglob("*")
    if path.is_file() and path.parent != MANIFEST_DIR
)
rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
    for path in artifact_paths
]
MANIFEST_CSV = MANIFEST_DIR / "F30K_BASE_MANIFEST.csv"
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(rows)
MANIFEST_JSON = MANIFEST_DIR / "F30K_BASE_MANIFEST.json"
write_json(MANIFEST_JSON, {
    "schema_version": "f30k_base_manifest_v1",
    "created_at_utc": now_utc(),
    "artifact_count": len(rows),
    "artifacts": rows,
})

for row in rows:
    path = OUTPUT_ROOT / row["relative_path"]
    assert path.stat().st_size == row["size_bytes"]
    assert sha256(path) == row["sha256"]

print("=" * 72)
print("F30K BASELINE PREFLIGHT: AUTHORIZED")
print("Baselines                 : T1 T2 T3 S1 S2 S3")
print("Development seed          : 42")
print("Max epochs / Early stop   : 20 / patience 4 after epoch 5")
print("Selection                 : Validation CIDEr-D")
print("Validation references     :", len(annotations))
print("Test status               : SEALED / NOT USED")
print("Output                    :", OUTPUT_ROOT)


F30K BASELINE PREFLIGHT: AUTHORIZED
Baselines                 : T1 T2 T3 S1 S2 S3
Development seed          : 42
Max epochs / Early stop   : 20 / patience 4 after epoch 5
Selection                 : Validation CIDEr-D
Validation references     : 5070
Test status               : SEALED / NOT USED
Output                    : /kaggle/working/F30K_Base
